In [1]:
import pandas as pd 
import json 
import lightgbm as lgb
from xgboost import XGBRegressor
import os
import logging
import inspect
import sys
from lightgbm import LGBMRegressor
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean


In [2]:
from sklearn.ensemble import RandomForestRegressor

In [3]:
import numpy as np

In [4]:

data_path = 'forecasting-data-set.parquet'
columns_json_path = 'forecast_set_columns.json'

In [5]:

df = pd.read_parquet(data_path)

 
with open(columns_json_path, 'r') as f:
    columns_dict = json.load(f)


In [6]:
def Prepare_Data():
    try:

        Id = columns_dict.get('Id',["unique_id"])
        Datestamp = columns_dict.get('Datestamp',["datestamp"])
        Target = columns_dict.get('Target',['y'])
        Exog = columns_dict.get('ExogenousFeature',[])
        #No Exog Columns for now

        data = df[Id+Datestamp+Target+Exog]
        data = data.rename(columns={'datestamp':'ds'})
        data['ds'] = pd.to_datetime(data['ds'])
        data = data.sort_values(by=['unique_id','ds'], ascending=True)


        return data

    except Exception as e:
        return pd.DataFrame()

In [7]:
data = Prepare_Data()

In [8]:
data

,unique_id,ds,y,month_in_quarter_sin,month_in_quarter_cos,month_sin,month_cos,quarter_sin,quarter_cos,week,week_sin,week_cos
4944,1,2000-01-01,12.9,8.660254e-01,-0.5,5.000000e-01,8.660254e-01,1.000000e+00,6.123234e-17,52,6.432491e-16,1.000000e+00
4945,1,2000-02-01,14.2,-8.660254e-01,-0.5,8.660254e-01,5.000000e-01,1.000000e+00,6.123234e-17,5,5.680647e-01,8.229839e-01
4946,1,2000-03-01,12.0,-2.449294e-16,1.0,1.000000e+00,6.123234e-17,1.000000e+00,6.123234e-17,9,8.854560e-01,4.647232e-01
4947,1,2000-04-01,12.2,8.660254e-01,-0.5,8.660254e-01,-5.000000e-01,1.224647e-16,-1.000000e+00,13,1.000000e+00,-1.608123e-16
4948,1,2000-05-01,11.8,-8.660254e-01,-0.5,5.000000e-01,-8.660254e-01,1.224647e-16,-1.000000e+00,18,8.229839e-01,-5.680647e-01
...,...,...,...,...,...,...,...,...,...,...,...,...
69520,554,2025-05-01,18.9,-8.660254e-01,-0.5,5.000000e-01,-8.660254e-01,1.224647e-16,-1.000000e+00,18,8.229839e-01,-5.680647e-01
69521,554,2025-06-01,21.8,-2.449294e-16,1.0,1.224647e-16,-1.000000e+00,1.224647e-16,-1.000000e+00,22,4.647232e-01,-8.854560e-01
69522,554,2025-07-01,22.3,8.660254e-01,-0.5,-5.000000e-01,-8.660254e-01,-1.000000e+00,-1.836970e-16,27,-1.205367e-01,-9.927089e-01
69523,554,2025-08-01,24.5,-8.660254e-01,-0.5,-8.660254e-01,-5.000000e-01,-1.000000e+00,-1.836970e-16,31,-5.680647e-01,-8.229839e-01


In [9]:
data['y'] = data.groupby(['unique_id'])['y'].transform(lambda x: x.interpolate())

In [ ]:
#Setup LGBM Model 


In [10]:
lgbmModel = LGBMRegressor()
xgbModel = XGBRegressor()


In [11]:
rfModel = RandomForestRegressor()

In [14]:
fsct = MLForecast(
                    models=[lgbmModel],
                    freq='MS',
                    lags=[1,2,3,4,10,11,12],
                    lag_transforms={
                        1:[RollingMean(window_size=4, min_samples=4)],
                        2:[RollingMean(window_size=4, min_samples=4)],
                        3:[RollingMean(window_size=4, min_samples=4)],
                        4:[RollingMean(window_size=4, min_samples=4)],
                    }   
                )

In [15]:
#In Built Cross Validation
cv_df = fsct.cross_validation(
                    df=data,
                    h=48,           # 4 year forecast horizon => 48 Months
                    n_windows=2,      # produce 3 forecasts => 3 Cross Validations 
                    step_size=48,     # move forward 4 years each window => move the training window 4 years later 
                    static_features=[]
)
            

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000635 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2917
[LightGBM] [Info] Number of data points in the train set: 111354, number of used features: 20
[LightGBM] [Info] Start training from score 78.775893
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001228 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2917
[LightGBM] [Info] Number of data points in the train set: 137946, number of used features: 20
[LightGBM] [Info] Start training from score 80.719744


In [16]:
cv_df

,unique_id,ds,cutoff,y,LGBMRegressor
0,1,2017-10-01,2017-09-01,20.8,20.429164
1,1,2017-11-01,2017-09-01,19.8,20.175507
2,1,2017-12-01,2017-09-01,20.4,19.990353
3,1,2018-01-01,2017-09-01,19.6,20.048320
4,1,2018-02-01,2017-09-01,18.3,20.099906
...,...,...,...,...,...
53179,554,2025-05-01,2021-09-01,18.9,19.037906
53180,554,2025-06-01,2021-09-01,21.8,19.002123
53181,554,2025-07-01,2021-09-01,22.3,18.893248
53182,554,2025-08-01,2021-09-01,24.5,18.877228


In [17]:
print(cv_df)

       unique_id         ds     cutoff     y  LGBMRegressor
0              1 2017-10-01 2017-09-01  20.8      20.429164
1              1 2017-11-01 2017-09-01  19.8      20.175507
2              1 2017-12-01 2017-09-01  20.4      19.990353
3              1 2018-01-01 2017-09-01  19.6      20.048320
4              1 2018-02-01 2017-09-01  18.3      20.099906
...          ...        ...        ...   ...            ...
53179        554 2025-05-01 2021-09-01  18.9      19.037906
53180        554 2025-06-01 2021-09-01  21.8      19.002123
53181        554 2025-07-01 2021-09-01  22.3      18.893248
53182        554 2025-08-01 2021-09-01  24.5      18.877228
53183        554 2025-09-01 2021-09-01  25.7      18.354382

[53184 rows x 5 columns]


In [18]:
from sklearn.metrics import root_mean_squared_error

In [19]:
grouped = cv_df.groupby(['unique_id', 'cutoff']).apply(
    lambda x: pd.Series({
        'RMSE': root_mean_squared_error(x['y'], x['LGBMRegressor']),
        'Avg_y': x['y'].mean()
    })
).reset_index()

grouped

C:\Users\tejas\AppData\Local\Temp\ipykernel_13616\3133144200.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped = cv_df.groupby(['unique_id', 'cutoff']).apply(


,unique_id,cutoff,RMSE,Avg_y
0,1,2017-09-01,2.437650,19.750000
1,1,2021-09-01,1.503911,22.497917
2,2,2017-09-01,0.948860,3.287500
3,2,2021-09-01,1.237916,4.525000
4,3,2017-09-01,32.957405,120.393750
...,...,...,...,...
1103,552,2021-09-01,15.658431,238.718750
1104,553,2017-09-01,4.092405,23.285417
1105,553,2021-09-01,2.025114,27.416667
1106,554,2017-09-01,4.052884,15.529167


In [20]:
grouped['cv_rmse'] = grouped['RMSE']/grouped['Avg_y']
grouped

,unique_id,cutoff,RMSE,Avg_y,cv_rmse
0,1,2017-09-01,2.437650,19.750000,0.123425
1,1,2021-09-01,1.503911,22.497917,0.066847
2,2,2017-09-01,0.948860,3.287500,0.288627
3,2,2021-09-01,1.237916,4.525000,0.273573
4,3,2017-09-01,32.957405,120.393750,0.273747
...,...,...,...,...,...
1103,552,2021-09-01,15.658431,238.718750,0.065594
1104,553,2017-09-01,4.092405,23.285417,0.175750
1105,553,2021-09-01,2.025114,27.416667,0.073864
1106,554,2017-09-01,4.052884,15.529167,0.260985


In [23]:
average_cv_per_cutoff = grouped.groupby(['cutoff']).apply(
    lambda x: pd.Series(
        {
            'avg_cv_rmse': x['cv_rmse'].mean()
        }
    )
).reset_index()
average_cv_per_cutoff


C:\Users\tejas\AppData\Local\Temp\ipykernel_13616\1851608477.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  average_cv_per_cutoff = grouped.groupby(['cutoff']).apply(


,cutoff,avg_cv_rmse
0,2017-09-01,0.235862
1,2021-09-01,0.215593


In [25]:
print(average_cv_per_cutoff)

      cutoff  avg_cv_rmse
0 2017-09-01     0.235862
1 2021-09-01     0.215593


In [27]:
weights = [0.2, 0.5]

average_cv_per_cutoff['weighted_avg_cv_rmse'] = average_cv_per_cutoff['avg_cv_rmse'] * weights
average_cv_per_cutoff

,cutoff,avg_cv_rmse,weighted_avg_cv_rmse
0,2017-09-01,0.235862,0.047172
1,2021-09-01,0.215593,0.107796


In [24]:
cutoff_cv_rmse_dict = average_cv_per_cutoff.set_index('cutoff')['avg_cv_rmse'].to_dict()
cutoff_cv_rmse_dict

{Timestamp('2017-09-01 00:00:00'): 0.23586224330439093,
 Timestamp('2021-09-01 00:00:00'): 0.21559272340201063}

In [77]:
cv_df.to_parquet('ml_models_cross_validation.parquet',index=False)

In [26]:

fsct.fit(data,
         static_features=[]
         )

MLForecast(models=[LGBMRegressor], freq=MS, lag_features=['lag1', 'lag2', 'lag3', 'lag4', 'lag10', 'lag11', 'lag12', 'rolling_mean_lag1_window_size4_min_samples4', 'rolling_mean_lag2_window_size4_min_samples4', 'rolling_mean_lag3_window_size4_min_samples4', 'rolling_mean_lag4_window_size4_min_samples4'], date_features=[], num_threads=1)

In [27]:
futureframe = fsct.make_future_dataframe(h=48)

In [30]:
futureframe

,unique_id,ds
0,1,2025-10-01
1,1,2025-11-01
2,1,2025-12-01
3,1,2026-01-01
4,1,2026-02-01
...,...,...
26587,554,2029-05-01
26588,554,2029-06-01
26589,554,2029-07-01
26590,554,2029-08-01


In [37]:
#Generating Exogenous Features 
futureframe['ds'] = pd.to_datetime(futureframe['ds'])

# Month and quarter
futureframe['month'] = futureframe['ds'].dt.month        # 1-12
futureframe['quarter'] = futureframe['ds'].dt.quarter    # 1-4

# Month within quarter: 1,2,3
futureframe['month_in_quarter'] = ((futureframe['month'] - 1) % 3) + 1

# Optional: cyclic encoding for month_in_quarter
futureframe['month_in_quarter_sin'] = np.sin(2 * np.pi * futureframe['month_in_quarter'] / 3)
futureframe['month_in_quarter_cos'] = np.cos(2 * np.pi * futureframe['month_in_quarter'] / 3)

# Cyclical encoding for month (for full year seasonality)
futureframe['month_sin'] = np.sin(2 * np.pi * futureframe['month'] / 12)
futureframe['month_cos'] = np.cos(2 * np.pi * futureframe['month'] / 12)

# Cyclical encoding for quarter (for quarterly seasonality)
futureframe['quarter_sin'] = np.sin(2 * np.pi * futureframe['quarter'] / 4)
futureframe['quarter_cos'] = np.cos(2 * np.pi * futureframe['quarter'] / 4)

futureframe['week'] = futureframe['ds'].dt.isocalendar().week.astype(int)


futureframe['week_sin'] = np.sin(2 * np.pi * futureframe['week'] / 52)
futureframe['week_cos'] = np.cos(2 * np.pi * futureframe['week'] / 52)

In [38]:
futureframe.columns

Index(['unique_id', 'ds', 'month_in_quarter_sin', 'month_in_quarter_cos',
       'month_sin', 'month_cos', 'quarter_sin', 'quarter_cos', 'week',
       'week_sin', 'month', 'quarter', 'month_in_quarter', 'week_cos'],
      dtype='object')

In [39]:
futureframe = futureframe.drop(['month', 'quarter', 'month_in_quarter'], axis=1)
futureframe

,unique_id,ds,month_in_quarter_sin,month_in_quarter_cos,month_sin,month_cos,quarter_sin,quarter_cos,week,week_sin,week_cos
0,1,2025-10-01,8.660254e-01,-0.5,-8.660254e-01,5.000000e-01,-2.449294e-16,1.000000e+00,40,-9.927089e-01,0.120537
1,1,2025-11-01,-8.660254e-01,-0.5,-5.000000e-01,8.660254e-01,-2.449294e-16,1.000000e+00,44,-8.229839e-01,0.568065
2,1,2025-12-01,-2.449294e-16,1.0,-2.449294e-16,1.000000e+00,-2.449294e-16,1.000000e+00,49,-3.546049e-01,0.935016
3,1,2026-01-01,8.660254e-01,-0.5,5.000000e-01,8.660254e-01,1.000000e+00,6.123234e-17,1,1.205367e-01,0.992709
4,1,2026-02-01,-8.660254e-01,-0.5,8.660254e-01,5.000000e-01,1.000000e+00,6.123234e-17,5,5.680647e-01,0.822984
...,...,...,...,...,...,...,...,...,...,...,...
26587,554,2029-05-01,-8.660254e-01,-0.5,5.000000e-01,-8.660254e-01,1.224647e-16,-1.000000e+00,18,8.229839e-01,-0.568065
26588,554,2029-06-01,-2.449294e-16,1.0,1.224647e-16,-1.000000e+00,1.224647e-16,-1.000000e+00,22,4.647232e-01,-0.885456
26589,554,2029-07-01,8.660254e-01,-0.5,-5.000000e-01,-8.660254e-01,-1.000000e+00,-1.836970e-16,26,-3.216245e-16,-1.000000
26590,554,2029-08-01,-8.660254e-01,-0.5,-8.660254e-01,-5.000000e-01,-1.000000e+00,-1.836970e-16,31,-5.680647e-01,-0.822984


In [40]:
futureforecasts = fsct.predict(
    h=48,
    X_df=futureframe
)

In [41]:
FullSet = pd.concat([cv_df, futureforecasts], axis=0)
FullSet

,unique_id,ds,cutoff,y,LGBMRegressor
0,1,2013-10-01,2013-09-01,16.8,18.854992
1,1,2013-11-01,2013-09-01,17.8,19.001289
2,1,2013-12-01,2013-09-01,17.3,18.588391
3,1,2014-01-01,2013-09-01,16.3,17.997679
4,1,2014-02-01,2013-09-01,17.7,17.566758
...,...,...,...,...,...
26587,554,2029-05-01,NaT,NaN,21.677619
26588,554,2029-06-01,NaT,NaN,22.193333
26589,554,2029-07-01,NaT,NaN,22.517050
26590,554,2029-08-01,NaT,NaN,22.504980


In [42]:
cutoffs = sorted(FullSet['cutoff'].dropna().unique().tolist())
cutoffs

[Timestamp('2013-09-01 00:00:00'),
 Timestamp('2017-09-01 00:00:00'),
 Timestamp('2021-09-01 00:00:00')]

In [43]:
cutoffs_dict = {
    cutoffs[0]: "Validation 1",
    cutoffs[1]: "Validateion 2",
    cutoffs[2]: "Test"
}

In [44]:
FullSet['Set'] = FullSet['cutoff'].map(cutoffs_dict)
FullSet['Set'] = FullSet['Set'].fillna('Future Forecast')

FullSet

,unique_id,ds,cutoff,y,LGBMRegressor,Set
0,1,2013-10-01,2013-09-01,16.8,18.854992,Validation 1
1,1,2013-11-01,2013-09-01,17.8,19.001289,Validation 1
2,1,2013-12-01,2013-09-01,17.3,18.588391,Validation 1
3,1,2014-01-01,2013-09-01,16.3,17.997679,Validation 1
4,1,2014-02-01,2013-09-01,17.7,17.566758,Validation 1
...,...,...,...,...,...,...
26587,554,2029-05-01,NaT,NaN,21.677619,Future Forecast
26588,554,2029-06-01,NaT,NaN,22.193333,Future Forecast
26589,554,2029-07-01,NaT,NaN,22.517050,Future Forecast
26590,554,2029-08-01,NaT,NaN,22.504980,Future Forecast


In [49]:
import plotly.express as px
import plotly.graph_objects as go

In [55]:
Sample = FullSet[FullSet['unique_id']==554]
Sample

,unique_id,ds,cutoff,y,LGBMRegressor,Set
26544,554,2013-10-01,2013-09-01,11.6,8.885640,Validation 1
26545,554,2013-11-01,2013-09-01,11.3,9.131912,Validation 1
26546,554,2013-12-01,2013-09-01,11.9,9.201128,Validation 1
26547,554,2014-01-01,2013-09-01,11.9,9.671459,Validation 1
26548,554,2014-02-01,2013-09-01,12.8,10.392305,Validation 1
...,...,...,...,...,...,...
26587,554,2029-05-01,NaT,NaN,21.677619,Future Forecast
26588,554,2029-06-01,NaT,NaN,22.193333,Future Forecast
26589,554,2029-07-01,NaT,NaN,22.517050,Future Forecast
26590,554,2029-08-01,NaT,NaN,22.504980,Future Forecast


In [57]:
SampleFull = data[data['unique_id']==554]

In [58]:
Line = px.line(
    title='LGBM Cross Validation and Future Forecast Sample for Series 1',
    width=1500,
    height=900,
    template='ggplot2'
)

Line.add_trace(
    go.Scatter(
    x=SampleFull['ds'],
    y=SampleFull['y'],
    name='Actual Value',
    mode='lines',
    line=dict(color='#263C7A', width=3.5))
)

Line.add_trace(
    go.Scatter(
    x=Sample['ds'],
    y=Sample['LGBMRegressor'],
    name='Predicted Value',
    mode='lines',
    line=dict(color="#7A2626", width=3.5, dash='dash'))
)

Line.show()